# Notebook 5: FAISS Vector Indexing

## Project: Enterprise Document Intelligence Assistant using LLM and RAG

This is the fifth notebook of the project.

In Notebook 4, we generated dense vector embeddings for every document chunk.

In this notebook, we will store those embeddings inside a FAISS vector index.

FAISS allows fast similarity search over thousands or millions of vectors.

The goal of this notebook is to:

1. Load the saved chunk embeddings.
2. Load the chunk metadata.
3. Build a FAISS vector index.
4. Add embeddings to the FAISS index.
5. Test semantic search using a sample query.
6. Save the FAISS index for the retrieval notebook.

The output file from this notebook will be:

`bbc_faiss_index.index`

This file will be used in:

`06_Retrieval_System.ipynb`

In [1]:
# Install required libraries
!pip install faiss-cpu sentence-transformers -q

# Import required libraries
import os
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


# Find input file automatically
def find_input_file(file_name):
    """
    Searches for a file in common Kaggle locations.

    This is useful because outputs from previous notebooks must usually
    be uploaded as input files in the next Kaggle notebook.
    """
    possible_paths = [
        file_name,
        f"/kaggle/working/{file_name}"
    ]

    # Search inside Kaggle input folders
    for root, dirs, files in os.walk("/kaggle/input"):
        if file_name in files:
            possible_paths.append(os.path.join(root, file_name))

    for path in possible_paths:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        f"{file_name} not found. Please upload it as input to this notebook."
    )


# Load embeddings and metadata
def load_inputs(embedding_file, metadata_file):
    """
    Loads embeddings and metadata generated in Notebook 4.
    """
    embedding_path = find_input_file(embedding_file)
    metadata_path = find_input_file(metadata_file)

    embeddings = np.load(embedding_path)
    metadata = pd.read_csv(metadata_path)

    print("Inputs loaded successfully.")
    print("Embedding file:", embedding_path)
    print("Metadata file:", metadata_path)
    print("Embeddings shape:", embeddings.shape)
    print("Metadata shape:", metadata.shape)

    return embeddings, metadata


# Build FAISS index
def build_faiss_index(embeddings):
    """
    Builds a FAISS index using inner product similarity.

    Since the embeddings are normalized, inner product is equivalent
    to cosine similarity.
    """
    embedding_dim = embeddings.shape[1]

    index = faiss.IndexFlatIP(embedding_dim)
    index.add(embeddings.astype("float32"))

    print("FAISS index created successfully.")
    print("Embedding dimension:", embedding_dim)
    print("Number of vectors in index:", index.ntotal)

    return index


# Save FAISS index
def save_faiss_index(index, output_file):
    """
    Saves the FAISS index to disk.
    """
    faiss.write_index(index, output_file)

    print("FAISS index saved successfully.")
    print("Output file:", output_file)


# Test semantic search
def test_search(query, model, index, metadata, top_k=5):
    """
    Runs a sample semantic search using the FAISS index.
    """
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    print("Query:", query)
    print("\nTop Retrieved Chunks:\n")

    results = []

    for rank, idx in enumerate(indices[0], start=1):
        row = metadata.iloc[idx]
        score = scores[0][rank - 1]

        result = {
            "rank": rank,
            "score": score,
            "chunk_id": row["chunk_id"],
            "title": row["title"],
            "category": row["category"],
            "chunk_text": row["chunk_text"]
        }

        results.append(result)

        print("=" * 80)
        print("Rank:", rank)
        print("Score:", round(float(score), 4))
        print("Chunk ID:", row["chunk_id"])
        print("Title:", row["title"])
        print("Category:", row["category"])
        print("\nChunk Preview:\n")
        print(row["chunk_text"][:700])
        print()

    return pd.DataFrame(results)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.5 MB/s eta 0:00:00:00:0100:01


In [2]:
# Load embeddings and metadata from Notebook 4
embedding_file = "bbc_chunk_embeddings.npy"
metadata_file = "bbc_chunk_metadata.csv"

embeddings, metadata = load_inputs(embedding_file, metadata_file)


# Preview metadata
print("\nMetadata Preview:")
display(metadata.head())


# Validate embeddings and metadata
if len(embeddings) != len(metadata):
    raise ValueError("Number of embeddings does not match number of metadata rows.")

print("\nValidation successful.")
print("Each embedding has a matching metadata row.")


# Build FAISS vector index
faiss_index = build_faiss_index(embeddings)


# Load embedding model for testing search
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("\nEmbedding model loaded for query testing.")
print("Model name:", MODEL_NAME)


# Test semantic search
sample_query = "latest developments in the economy and business markets"

search_results = test_search(
    query=sample_query,
    model=embedding_model,
    index=faiss_index,
    metadata=metadata,
    top_k=5
)

print("\nSearch Results Table:")
display(search_results[["rank", "score", "chunk_id", "title", "category"]])


# Save FAISS index
output_index_file = "bbc_faiss_index.index"

save_faiss_index(faiss_index, output_index_file)


# Verify saved FAISS index
loaded_index = faiss.read_index(output_index_file)

print("\nSaved FAISS index verified successfully.")
print("Number of vectors in loaded index:", loaded_index.ntotal)

Inputs loaded successfully.
Embedding file: /kaggle/input/datasets/jahnavidulala/bbc-chunk-embeddings-and-metadata/bbc_chunk_embeddings.npy
Metadata file: /kaggle/input/datasets/jahnavidulala/bbc-chunk-embeddings-and-metadata/bbc_chunk_metadata.csv
Embeddings shape: (8622, 384)
Metadata shape: (8622, 7)

Metadata Preview:


,chunk_id,doc_id,chunk_index,title,category,chunk_text,chunk_word_count
0,1_1,1,1,Ukraine conflict: Your guide to understanding ...,unknown,More than 1.5 million Ukrainians have fled the...,21
1,2_1,2,1,Russian gymnast investigated for wearing pro-w...,unknown,Russian gymnast Ivan Kuliak is being investiga...,31
2,3_1,3,1,Ukraine crisis: The West fights back against P...,unknown,Several US presidents have failed to get the m...,21
3,4_1,4,1,Ukraine maps: New agreed ceasefire breaks down...,unknown,A ceasefire agreement in the southern city of ...,20
4,5_1,5,1,Man in dinghy in near miss with Southampton-bo...,unknown,The moment a man swims out of the path of a co...,20



Validation successful.
Each embedding has a matching metadata row.
FAISS index created successfully.
Embedding dimension: 384
Number of vectors in index: 8622


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded for query testing.
Model name: sentence-transformers/all-MiniLM-L6-v2
Query: latest developments in the economy and business markets

Top Retrieved Chunks:

Rank: 1
Score: 0.4727
Chunk ID: 158_1
Title: BBC editors react to Sunak's 2022 Spring Statement
Category: unknown

Chunk Preview:

Laura Kuenssberg, Faisal Islam and Simon Jack review the headlines and reaction to the chancellor's update on the UK economy.

Rank: 2
Score: 0.4026
Chunk ID: 1874_1
Title: Australia challenges China in mining for essential elements
Category: unknown

Chunk Preview:

It is an industry that has been dominated by China for years, but now the rest of the world wants in.

Rank: 3
Score: 0.3908
Chunk ID: 5128_1
Title: 'We failed - why our dream eco-business collapsed'
Category: unknown

Chunk Preview:

With a big rise in the number of firms going bust, we speak to one couple about why their business folded.

Rank: 4
Score: 0.3896
Chunk ID: 1194_1
Title: IMF: UK set for slowest growth 

,rank,score,chunk_id,title,category
0,1,0.472734,158_1,BBC editors react to Sunak's 2022 Spring State...,unknown
1,2,0.402591,1874_1,Australia challenges China in mining for essen...,unknown
2,3,0.390758,5128_1,'We failed - why our dream eco-business collap...,unknown
3,4,0.389637,1194_1,IMF: UK set for slowest growth of G7 countries...,unknown
4,5,0.372572,1042_1,Retail sales fall at fastest rate since lockdown,unknown


FAISS index saved successfully.
Output file: bbc_faiss_index.index

Saved FAISS index verified successfully.
Number of vectors in loaded index: 8622
